# Langfuse basics - tracing a LangChain agent

An agent makes many hidden decisions: which tool to call, with what arguments, what the tool returned, how long each step took, what it cost. **Tracing** records all of that so you can see it instead of guessing.

In this notebook we build a small agent and trace it with [Langfuse](https://langfuse.com):

- a **supervisor agent** with two tools:
  - `web_search` - looks things up on the web,
  - `calculator` - a **sub-agent** that has four tools of its own: `add`, `subtract`, `multiply`, `divide`;
- every conversation turn becomes **one clean trace**, with every step nested inside it;
- every trace carries a **user id** and a **session id**, so you can follow one person and one conversation.

```
 supervisor agent
 ├── web_search
 └── calculator  (sub-agent)
      ├── add
      ├── subtract
      ├── multiply
      └── divide
```

**Needs:** `OPENAI_API_KEY`, `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY` (and `LANGFUSE_BASE_URL`) in your `.env`. Langfuse keys: *Project settings -> API keys* in your Langfuse project.

## 1. The three building blocks

Langfuse organises everything in three nested levels:

```
┌──────────────────────────────── SESSION ─────────────────────────────────┐
│  ┌──────── TRACE ────────┐  ┌──────── TRACE ────────┐  ┌──── TRACE ────┐ │
│  │  ● observation        │  │  ● observation        │  │ ● observation │ │
│  │  ● observation        │  │  ● observation        │  │ ● observation │ │
│  │  ● observation        │  │  ● observation        │  └───────────────┘ │
│  └───────────────────────┘  │  ● observation        │                    │
│                             └───────────────────────┘                    │
└──────────────────────────────────────────────────────────────────────────┘
```

| Level | What it is | In our agent |
|---|---|---|
| **Observation** | one step: an LLM call (`generation`), a tool call (`tool`), or any piece of work (`span`). Observations nest inside each other. | the model call, `web_search`, `calculator`, `multiply` ... |
| **Trace** | one complete request: a tree of observations. | one chat turn: question in, answer out |
| **Session** | a group of traces that belong together. | one whole conversation |

Two more attributes ride along on every trace: **`user_id`** (who) and **`session_id`** (which conversation).

In the **observations table** the observation columns differ per row, while the trace-level columns repeat on every row:

| id | type | name | latency | trace_id | trace_name | user_id | session_id |
|---|---|---|---|---|---|---|---|
| obs-1 | span | handle-chat | 3.1s | abc123 | chat-request | u-42 | s-7 |
| obs-2 | span | retrieval | 0.4s | abc123 | chat-request | u-42 | s-7 |
| obs-3 | generation | llm-call | 2.6s | abc123 | chat-request | u-42 | s-7 |
| obs-4 | span | build-report | 1.2s | def789 | export-report | u-17 | s-9 |
| obs-5 | generation | summarize | 0.9s | def789 | export-report | u-17 | s-9 |

*Observation-level data* = `id`, `type`, `name`, `latency`. *Trace-level data* = `trace_id`, `trace_name`, `user_id`, `session_id`, copied onto every row so you can filter any observation by user or session.

## 2. Setup

Load `.env` **before** creating the Langfuse client, otherwise it starts without credentials. `environment="workshop"` keeps these traces apart from real ones.

In [1]:
%pip install -qU langfuse langchain langchain-openai ddgs python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: E:\Academy Git\genai-weekday-batch1\Week 11\.venv\Scripts\python.exe -m pip install --upgrade pip


In [2]:
import os, getpass
from dotenv import load_dotenv

load_dotenv()

for key in ("OPENAI_API_KEY", "LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY"):
    if not os.environ.get(key):
        os.environ[key] = getpass.getpass(f"{key}: ")

if not os.environ.get("LANGFUSE_BASE_URL"):
    os.environ["LANGFUSE_BASE_URL"] = "https://cloud.langfuse.com"   # EU cloud; US is https://us.cloud.langfuse.com

from langfuse import Langfuse, propagate_attributes
from langfuse.langchain import CallbackHandler

langfuse = Langfuse(environment="workshop")
assert langfuse.auth_check(), "Langfuse authentication failed - check your keys and LANGFUSE_BASE_URL"
print("Langfuse connected ->", os.environ["LANGFUSE_BASE_URL"])

python-dotenv could not parse statement starting at line 11


Langfuse connected -> https://us.cloud.langfuse.com


## 3. The agent

### 3.1 Arithmetic tools

Four small tools. `divide` returns a message instead of raising on zero, so the agent can explain the problem.

In [3]:
from langchain_core.tools import tool


@tool
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b


@tool
def subtract(a: float, b: float) -> float:
    """Subtract b from a."""
    return a - b


@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b


@tool
def divide(a: float, b: float) -> float | str:
    """Divide a by b."""
    if b == 0:
        return "Error: division by zero is undefined."
    return a / b

### 3.2 The arithmetic sub-agent, wrapped as a tool

The sub-agent is an agent whose only job is arithmetic. We wrap it in a normal tool called `calculator`, so the supervisor can call it like any other tool.

Look at the signature: `calculator(task, config: RunnableConfig)`. LangChain fills `config` in automatically, and it carries the **tracing context**. We hand it to the sub-agent with `config=config`, so the sub-agent's steps nest *inside* the `calculator` observation. In plain synchronous code LangChain would often link them for you anyway - passing `config` explicitly is the habit that keeps working when it would not (section 7 shows when).

In [4]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnableConfig

llm = ChatOpenAI(model="gpt-5-nano")

arithmetic_agent = create_agent(
    model=llm,
    tools=[add, subtract, multiply, divide],
    system_prompt=(
        "You are an arithmetic assistant. Use the tools for every calculation step - never calculate in your head. "
        "Reply with the final result and one short line showing the working."
    ),
)


@tool
def calculator(task: str, config: RunnableConfig) -> str:
    """Solve an arithmetic task written in plain words, e.g. '15% of 2480 plus 120'. Handles add, subtract, multiply, divide."""
    result = arithmetic_agent.invoke({"messages": [HumanMessage(content=task)]}, config=config)   # config = keep the trace connected
    return result["messages"][-1].content

### 3.3 Web search tool

DuckDuckGo through `ddgs` - no API key needed.

In [5]:
from ddgs import DDGS


@tool
def web_search(query: str) -> str:
    """Search the web and return the top results (title, url, snippet)."""
    try:
        hits = DDGS().text(query, max_results=3)
    except Exception as e:
        return f"Search failed: {type(e).__name__}"
    if not hits:
        return "No results."
    return "\n\n".join(f"[{i}] {h['title']}\n{h['href']}\n{h['body']}" for i, h in enumerate(hits, 1))

### 3.4 The supervisor agent

In [6]:
agent = create_agent(
    model=llm,
    tools=[web_search, calculator],
    system_prompt=(
        "You are a concise assistant. Use web_search for facts you need to look up. "
        "Use calculator for ANY arithmetic - never calculate yourself. Keep answers short."
    ),
)

## 4. Step 1 - your first trace

The smallest possible instrumentation: create a `CallbackHandler` and pass it to `invoke` in `config={"callbacks": [...]}`. The LangChain integration records the LLM calls and tool calls automatically.

`langfuse.flush()` sends the buffered data now. Langfuse sends in the background, so short scripts and notebooks should flush before you go and look.

In [7]:
handler = CallbackHandler()
result = agent.invoke(
    {"messages": [HumanMessage(content="What is 15% of 2480?")]},
    config={"callbacks": [handler]},
)
print(result["messages"][-1].content)

langfuse.flush()
print("Trace:", langfuse.get_trace_url(trace_id=handler.last_trace_id))

372


Trace: https://us.cloud.langfuse.com/project/cmv804pom00e4ad0eexnnx3pi/traces/61302ce3714584e0bea34ba41c1251b5


Open the link. You will see a tree of observations that Langfuse created for you: the agent run, one `generation` per model call (with the **model name, token counts and cost**), and a `tool` observation per tool call - with the `calculator` call containing the sub-agent's own steps.

But look at the **top of the trace**: it is named after the framework (`LangGraph`), it has **no user and no session**, and its input / output are the raw message lists. Fine for a quick look, not clean. Steps 2 and 3 fix that.

## 5. Step 2 - one clean trace per turn

We wrap each chat turn in our own **root observation**, `handle-chat`:

- `start_as_current_observation(name=..., as_type="span", input=...)` opens it; everything created inside becomes its child, so the whole turn is **one trace**.
- We set the **input** to just the user's question and the **output** to just the final reply - what a reviewer wants to see at a glance.
- Names are **verb-first and fixed** (`handle-chat`). Anything dynamic, like a user's name, belongs in metadata, not in the name.

In [8]:
def ask_step2(question: str) -> str:
    with langfuse.start_as_current_observation(name="handle-chat", as_type="span", input=question) as root:
        result = agent.invoke(
            {"messages": [HumanMessage(content=question)]},
            config={"callbacks": [CallbackHandler()]},      # nests under the root observation automatically
        )
        reply = result["messages"][-1].content
        root.update(output=reply)
    langfuse.flush()
    print("Trace:", langfuse.get_trace_url(trace_id=root.trace_id))
    return reply


print(ask_step2("What is 15% of 2480?"))

Trace: https://us.cloud.langfuse.com/project/cmv804pom00e4ad0eexnnx3pi/traces/03848c1d2a255224380535996bba5e3e
372.0


Compare with Step 1: the trace now starts with a readable `handle-chat` root whose input is the question and whose output is the answer. Still no user or session - next.

## 6. Step 3 - user id and session id

`propagate_attributes(...)` stamps attributes on **every observation created inside it**:

- `user_id` - who is asking (powers the **Users** page),
- `session_id` - which conversation (powers the **Sessions** page),
- `trace_name` - the name shown for the whole trace.

Two rules: put it **at the start of the turn** (observations created before it do not get the attributes), and use the **same `session_id` for every turn of one conversation**.

`ChatSession` below keeps a `user_id`, a `session_id` and a short history. Each `say()` is one turn = one trace; all turns of a `ChatSession` share the session id.

In [9]:
import uuid

RUN_ID = uuid.uuid4().hex[:6]      # keeps session ids unique per run so reruns do not mix together


def ask(question: str, user_id: str, session_id: str, history=()) -> str:
    with langfuse.start_as_current_observation(name="handle-chat", as_type="span", input=question) as root:
        with propagate_attributes(user_id=user_id, session_id=session_id, trace_name="chat-request"):
            result = agent.invoke(
                {"messages": [*history, HumanMessage(content=question)]},
                config={"callbacks": [CallbackHandler()]},
            )
            reply = result["messages"][-1].content
            root.update(output=reply)
    langfuse.flush()
    print(f"   trace: {langfuse.get_trace_url(trace_id=root.trace_id)}")
    return reply


class ChatSession:
    """One conversation: fixed user + session, plus the message history so the agent remembers earlier turns."""

    def __init__(self, user_id: str, name: str):
        self.user_id = user_id
        self.session_id = f"{name}-{RUN_ID}"
        self.history = []

    def say(self, question: str) -> None:
        print(f"[{self.user_id}] You: {question}")
        reply = ask(question, self.user_id, self.session_id, self.history)
        self.history += [HumanMessage(content=question), AIMessage(content=reply)]
        print(f"Agent: {reply}\n")

## 7. Step 4 - checking that sub-agent steps nest cleanly

A tool that runs another agent is where traces most often go wrong, so let's look at ours. One question that needs the calculator:

In [10]:
question = "What is (17 * 23) minus 140?"

with langfuse.start_as_current_observation(name="handle-chat", as_type="span", input=question) as root:
    with propagate_attributes(user_id="teacher", session_id=f"nesting-demo-{RUN_ID}", trace_name="chat-request"):
        out = agent.invoke({"messages": [HumanMessage(content=question)]}, config={"callbacks": [CallbackHandler()]})
        root.update(output=out["messages"][-1].content)
langfuse.flush()
print(out["messages"][-1].content)
print("Trace:", langfuse.get_trace_url(trace_id=root.trace_id))

251
Trace: https://us.cloud.langfuse.com/project/cmv804pom00e4ad0eexnnx3pi/traces/d5aac1456e6d7320142013ed6a4ffec0


Open the trace. Every step sits where it belongs - the sub-agent's work is **inside** the `calculator` observation:

```
handle-chat
└── LangGraph                      (supervisor)
    ├── model  -> ChatOpenAI       (decides to call calculator)
    ├── tools
    │   └── calculator             (tool)
    │       └── LangGraph          (arithmetic sub-agent)
    │           ├── model -> ChatOpenAI
    │           ├── tools -> multiply
    │           ├── model -> ChatOpenAI
    │           ├── tools -> subtract
    │           └── model -> ChatOpenAI
    └── model  -> ChatOpenAI       (writes the final answer)
```

### Why we pass `config` explicitly

In plain synchronous code LangChain carries the tracing context along automatically, so even a sub-agent called *without* `config` usually still nests. That is fragile: the link is kept in Python's "current context", which is **not** passed to another thread (and on Python older than 3.11 not through `async` calls either). Run the sub-agent in a worker thread - say to add a timeout - and, without `config`, it silently drops out of the trace. Same question, two thread-based calculators:

In [11]:
from concurrent.futures import ThreadPoolExecutor


def run_in_thread(task, config=None):
    with ThreadPoolExecutor(max_workers=1) as pool:
        result = pool.submit(arithmetic_agent.invoke, {"messages": [HumanMessage(content=task)]}, config).result()
    return result["messages"][-1].content


@tool("calculator")
def calculator_thread_no_config(task: str) -> str:
    """Solve an arithmetic task written in plain words, e.g. '15% of 2480 plus 120'."""
    return run_in_thread(task)                    # no config: the trace link is lost in the new thread


@tool("calculator")
def calculator_thread_with_config(task: str, config: RunnableConfig) -> str:
    """Solve an arithmetic task written in plain words, e.g. '15% of 2480 plus 120'."""
    return run_in_thread(task, config)            # config carried across: the trace link survives


for label, calc in [("NO config  ", calculator_thread_no_config), ("WITH config", calculator_thread_with_config)]:
    thread_agent = create_agent(model=llm, tools=[calc], system_prompt="Use calculator for ANY arithmetic. Keep answers short.")
    with langfuse.start_as_current_observation(name="handle-chat", as_type="span", input=question) as root:
        with propagate_attributes(user_id="teacher", session_id=f"nesting-demo-{RUN_ID}", trace_name="chat-request"):
            out = thread_agent.invoke({"messages": [HumanMessage(content=question)]}, config={"callbacks": [CallbackHandler()]})
            root.update(output=out["messages"][-1].content)
    langfuse.flush()
    print(f"{label}: {langfuse.get_trace_url(trace_id=root.trace_id)}")

NO config  : https://us.cloud.langfuse.com/project/cmv804pom00e4ad0eexnnx3pi/traces/3778febdd4291eef6d822b1720c03524


WITH config: https://us.cloud.langfuse.com/project/cmv804pom00e4ad0eexnnx3pi/traces/31a078a160ed2aa90e67a7182c3c03d8


Compare the two traces: with **NO config** the `calculator` observation is a single leaf - you only learn "calculator took 4 seconds", not which operations it ran. With **WITH config** the `multiply` and `subtract` calls are back in the trace - as a branch right next to the `calculator` observation (under `tools`). Passing `config` costs nothing and keeps the trace complete whatever the surrounding code does.

## 8. Run real conversations

Two users, two sessions, a few turns each. Alice mostly does maths; Bob searches the web and mixes in a calculation. Every turn prints a link to its trace.

In [12]:
alice = ChatSession("alice", "alice-maths")
alice.say("What is 15% of 2480?")
alice.say("Now add 320 to that and divide the total by 4.")
alice.say("Which is bigger: 17 times 23, or 400 minus 12?")

[alice] You: What is 15% of 2480?


   trace: https://us.cloud.langfuse.com/project/cmv804pom00e4ad0eexnnx3pi/traces/e16bbe09677034eb959f92b1386e29ce
Agent: 372

[alice] You: Now add 320 to that and divide the total by 4.


   trace: https://us.cloud.langfuse.com/project/cmv804pom00e4ad0eexnnx3pi/traces/a13461f315e75ed4357b49f11b049755
Agent: 173

[alice] You: Which is bigger: 17 times 23, or 400 minus 12?


   trace: https://us.cloud.langfuse.com/project/cmv804pom00e4ad0eexnnx3pi/traces/0201ced3304824c0e562a940bd7b95db
Agent: 17 × 23 = 391; 400 − 12 = 388. So 17 × 23 is bigger (by 3).



In [13]:
bob = ChatSession("bob", "bob-research")
bob.say("Search the web: who created the Python programming language, and in what year was it first released?")
bob.say("How many years ago was that, if the current year is 2026?")
bob.say("What is 7 divided by 0?")

[bob] You: Search the web: who created the Python programming language, and in what year was it first released?


   trace: https://us.cloud.langfuse.com/project/cmv804pom00e4ad0eexnnx3pi/traces/66a022582a3121156c7e7df30d6272ea
Agent: Guido van Rossum created Python, and it was first released in 1991.

[bob] You: How many years ago was that, if the current year is 2026?


   trace: https://us.cloud.langfuse.com/project/cmv804pom00e4ad0eexnnx3pi/traces/3c72f57a0b2a3bf585bbd7b000a3538d
Agent: 35 years ago.

[bob] You: What is 7 divided by 0?


   trace: https://us.cloud.langfuse.com/project/cmv804pom00e4ad0eexnnx3pi/traces/e35a477e34f664fa577fadf85b35696c
Agent: Undefined. Division by zero is not defined.



## 9. Tour of Langfuse

Open your Langfuse project and look at the three pages from the sidebar.

**Tracing** - one row per trace (`chat-request`), or switch to the **observations** view for one row per step.
- Click a trace and expand the tree. Click `calculator` to see its input (the task) and output.
- Click a `generation`: model name, input / output messages, token counts, cost, latency.
- Filter by `user_id`, `session_id` or `environment = workshop`; sort by latency to find the slowest step.
- In the observations view you will see exactly the table from section 1: per-step columns plus `trace_name`, `user_id`, `session_id` repeated on every row.

**Sessions** - one row per conversation.
- Open `alice-maths-...`: her three turns in order, with total latency and cost.
- Open `bob-research-...`: the web search turn, the follow-up and the divide-by-zero turn.

**Users** - one row per `user_id`.
- `alice` and `bob` with their number of traces and total cost; click a user to see all of their traces.

## 10. Recap and exercises

**What we did**
1. `CallbackHandler` in `config=` gives you traces of LLM and tool calls for free.
2. A root observation (`start_as_current_observation`) makes each turn **one trace** with a clean input and output.
3. `propagate_attributes(user_id, session_id, trace_name)` puts the user and session on every observation.
4. Passing `config` into a sub-agent keeps its steps **nested** instead of lost.
5. `flush()` when your program is short-lived.

**Try it yourself**
- Add `tags=["maths"]` to `propagate_attributes` and filter by it in Tracing.
- Add `metadata={"channel": "notebook"}` and filter on it.
- Score a trace: inside the root observation call `langfuse.score_current_trace(name="helpful", value=1)`.
- Add a fifth arithmetic tool (`power`) and check that it appears under `calculator`.
- Break something on purpose (ask for `7 / 0`) and find the error in the trace.

**Next:** `ticket_agent_langfuse.ipynb` applies the same ideas to a larger agent - guardrails, memory, masking sensitive data, environments and release versions.